# Reinforcement Learning

# 6. Bandit Algorithms

This notebook presents **multi-armed bandit** algorithms.

Please read the instructions:
* Write concise code and text.
* Check that your notebook runs without errors.
* Clear all cell outputs before upload on Moodle.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
from model import Environment
from agent import Agent

## Multi-Armed Bandit

Multi-armed bandits are single-state models with random rewards.

In [ ]:
class MAB(Environment):
    """Multi-Armed Bandit environnement.
    
    Parameters
    ----------
    distribution : string
        Reward distribution (Bernoulli, Uniform or Gaussian)
    params : list
        List of parameters (one per action)
        Example for Bernoulli (mean): [0.4, 0.5, 0.6]
        Example for Uniform (low, high): [(-1, 1), (0, 1), (-1, 2)]
        Example for Gaussian (mean, variance): [(0, 1), (1, 2), (-1, 1)]
    """

    def __init__(self, distribution='bernoulli', params=[0.4, 0.6]):        
        if type(distribution) != str:
            raise ValueError("The parameter 'distribution' must be a string: either 'bernoulli', 'uniform' or 'gaussian'.")
        self.distribution = distribution.lower()
        self.params = params
         
    @staticmethod
    def get_states():
        """Single state."""
        return [None]
    
    def get_actions(self):
        """One action per arm."""
        actions = [action for action, _ in enumerate(self.params)]
        return actions
    
    def get_reward(self, action):
        """Random reward. The parameter depends on the action."""
        if self.distribution == 'bernoulli':
            return np.random.random() < self.params[action]
        if self.distribution == 'uniform':
            low, high = self.params[action]
            return np.random.uniform(low, high)
        if self.distribution in ['gaussian', 'normal']:
            mean, std = self.params[action]
            return np.random.normal(mean, std)
        raise ValueError('Unknown distribution.')
        
    def step(self, action):
        stop = False
        reward = self.get_reward(action)
        return reward, stop

In [ ]:
class Bandit(Agent):
    """Bandit algorithm with random policy. 

    Parameters
    ----------
    environment : object of class MAB
        Environment of the agent.
    init_value : float
        Initial value of the action-value function.
    init_count : int
        Initial count of the action-count function.
    """
    
    def __init__(self, environment, init_value=0, init_count=0):
        if not isinstance(environment, MAB):
            raise ValueError('The environment must be a multi-armed bandit.')
        self.environment = environment
        self.policy = self.random_policy
        actions = environment.get_actions()
        self.values = len(actions) * [init_value]
        self.counts = len(actions) * [init_count]

    def get_actions(self, state=None):
        """Get all possible actions."""
        return self.environment.get_actions()
    
    def get_episode(self, horizon=100):
        """Get the rewards for an episode and update the values."""
        state = None
        rewards = []
        for t in range(horizon):
            action = self.get_action(state)
            reward, _ = self.environment.step(action)
            rewards.append(reward)
            self.counts[action] += 1
            diff = reward - self.values[action]
            # update by temporal difference
            self.values[action] += diff / self.counts[action]
        return rewards    

In [ ]:
environment = MAB()

In [ ]:
environment.distribution

In [ ]:
environment.params

In [ ]:
agent = Bandit(environment)

In [ ]:
agent.get_actions()

In [ ]:
rewards = agent.get_episode(horizon=100)

In [ ]:
np.mean(rewards)

In [ ]:
agent.values

In [ ]:
agent.counts

## The $\varepsilon$-greedy policy

In [ ]:
class Greedy(Bandit):
    """Bandit algorithm with epsilon-greedy policy. 

    Parameters
    ----------
    environment : object of class MAB
        Environment of the agent.
    epsilon : float in [0, 1]
        Exploration rate.
    init_value : float
        Initial value of the action-value function.
    init_count : int
        Initial count of the action-count function.
    """
    
    def __init__(self, environment, epsilon=0.1, init_value=0, init_count=0):
        super(Greedy, self).__init__(environment, init_value, init_count) 
        self.epsilon = epsilon

    def get_action(self, state=None):
        """Get action with eps-greedy policy."""
        actions = self.get_actions()
        if np.random.random() > self.epsilon:
            # select the best action(s) with probability 1 - epsilon
            values = np.array(self.values)
            actions = np.flatnonzero(values==np.max(values))
        return np.random.choice(actions)


In [ ]:
agent = Greedy(environment)

In [ ]:
rewards = agent.get_episode(horizon=100)

In [ ]:
np.mean(rewards)

In [ ]:
agent.values

In [ ]:
agent.counts

## To do 

Consider a 2-armed bandit with Gaussian rewards of means $+1$ and $-1$, and unit variance.
* What is the expected return per action of the $\varepsilon$-greedy policy in steady state?
* Check your result by simulation. You might plot the evolution of the cumulative reward per action over time.

### Answer

Consider a 2-armed bandit with Gaussian rewards of means $\mu_1=+1$ and $\mu_2=-1$ (unit variance).

In steady state, an $\varepsilon$-greedy policy exploits the best arm with probability $1-\varepsilon$ and explores with probability $\varepsilon$ (uniform choice), hence:
- $P(\text{arm }+1) = (1-\varepsilon) + \varepsilon/2 = 1-\varepsilon/2$
- $P(\text{arm }-1) = \varepsilon/2$

Therefore the expected reward per step is:
$$
\mathbb{E}[R] = (1-\varepsilon/2)\cdot 1 + (\varepsilon/2)\cdot(-1) = 1-\varepsilon.
$$

In [ ]:
eps = 0.1
horizon = 20000
n_runs = 200

environment = MAB(distribution='gaussian', params=[(1, 1), (-1, 1)])

avg_reward_over_time = np.zeros(horizon)

for _ in range(n_runs):
    agent = Greedy(environment, epsilon=eps, init_value=0.0, init_count=0)
    rewards = agent.get_episode(horizon=horizon)
    rewards = np.array(rewards, dtype=float)
    cum_avg = np.cumsum(rewards) / (1 + np.arange(horizon))
    avg_reward_over_time += cum_avg

avg_reward_over_time /= n_runs

print("Théorie steady-state E[R] =", 1 - eps)
print("Dernière valeur simulée (moyenne cumulée) =", avg_reward_over_time[-1])

plt.figure()
plt.plot(avg_reward_over_time)
plt.xlabel("t")
plt.ylabel("moyenne cumulée des rewards")
plt.title(f"Epsilon-greedy (epsilon={eps}) sur bandit gaussien (+1, -1)")
plt.show()

### Remark

In simulation, the cumulative average reward converges to ≈0.9, matching the theoretical steady-state value 1−ε.

## To do

We now illustrate the principle of **optimism** in face of incertainty. 

We consider a 2-armed bandit with Bernoulli rewards with parameters $0.6$ and $0.5$ under the greedy policy (that is, $\varepsilon=0$).

* Show this principle by some suitable choices of the parameters ``init_value`` and ``init_count``.
* Comment the results.

In [ ]:
class PureGreedy(Bandit):
    """Pure greedy bandit (epsilon=0)."""

    def get_action(self, state=None):
        values = np.array(self.values, dtype=float)
        best = np.max(values)
        best_actions = np.flatnonzero(values == best)  
        return int(np.random.choice(best_actions))


environment = MAB(distribution='bernoulli', params=[0.6, 0.5])
horizon = 2000

agent_plain = PureGreedy(environment, init_value=0.0, init_count=0)
rewards_plain = np.array(agent_plain.get_episode(horizon=horizon), dtype=float)

agent_opt = PureGreedy(environment, init_value=1.0, init_count=0)
rewards_opt = np.array(agent_opt.get_episode(horizon=horizon), dtype=float)

avg_plain = np.cumsum(rewards_plain) / (1 + np.arange(horizon))
avg_opt   = np.cumsum(rewards_opt)   / (1 + np.arange(horizon))

print("No optimism:")
print("  counts =", agent_plain.counts, "values =", agent_plain.values, "mean =", avg_plain[-1])

print("Optimistic init:")
print("  counts =", agent_opt.counts, "values =", agent_opt.values, "mean =", avg_opt[-1])

plt.figure()
plt.plot(avg_plain, label="pure greedy, init_value=0, init_count=0")
plt.plot(avg_opt, label="pure greedy, optimistic init_value=1, init_count=0")
plt.xlabel("t")
plt.ylabel("cumulative average reward")
plt.legend()
plt.show()

### Remark

With pure greedy (ε=0), a non-optimistic start can get stuck on the first tried arm (here the 0.5 arm), while an optimistic initialization forces early exploration and quickly leads to the optimal arm (0.6), giving a higher long-run average reward.

## The UCB policy

We now consider the UCB (Upper Confidence Bound) policy.

## To do

* Complete and test the agent ``UCB`` below.
* Plot the evolution of the cumulative reward per action, and compare with that of $\varepsilon$-greedy policy for different values of $\varepsilon$.
* Repeat this experiment for the other models (uniform and Gaussian).<br> Interpret the results.

In [ ]:
class UCB(Bandit):
    """Bandit algorithm with UCB policy. 

    Parameters
    ----------
    environment : object of class MAB
        Environment of the agent.
    const : float in [0, 1]
        Multiplicative constant for the UCB bonus.
    init_value : float
        Initial value of the action-value function.
    init_count : int
        Initial count of the action-count function.
    """
    
    def __init__(self, environment, const=1, init_value=0, init_count=0):
        super(UCB, self).__init__(environment, init_value, init_count) 
        self.const = const

    def get_action(self, state=None):
        """Get action with UCB policy."""
        values = np.array(self.values)
        counts = np.array(self.counts)
        actions = self.get_actions()
        
        zero = np.flatnonzero(counts == 0)
        if len(zero) > 0:
            return int(np.random.choice(zero))

        t = int(np.sum(counts))

        bonus = self.const * np.sqrt(np.log(t) / counts)
        ucb_scores = values + bonus

        best = np.max(ucb_scores)
        best_actions = np.flatnonzero(ucb_scores == best)

        return np.random.choice(best_actions)

In [ ]:
agent = UCB(environment)

In [ ]:
def run_agent(agent_class, env_kwargs, agent_kwargs, horizon=5000, n_runs=200, seed=0):
    """
    Returns:
      avg_cum_reward[t] = average over runs of cumulative reward up to time t
      avg_cum_avg[t]    = average over runs of cumulative average reward up to time t
    """
    rng = np.random.default_rng(seed)
    cum_rewards = np.zeros(horizon)
    cum_avgs = np.zeros(horizon)

    for r in range(n_runs):
        np.random.seed(rng.integers(0, 2**32 - 1, dtype=np.uint32).item())

        env = MAB(**env_kwargs)
        agent = agent_class(env, **agent_kwargs)
        rewards = np.array(agent.get_episode(horizon=horizon), dtype=float)

        cr = np.cumsum(rewards)
        cum_rewards += cr
        cum_avgs += cr / (1 + np.arange(horizon))

    cum_rewards /= n_runs
    cum_avgs /= n_runs
    return cum_rewards, cum_avgs


def plot_comparison(env_kwargs, horizon=5000, n_runs=200, eps_list=(0.0, 0.01, 0.1), ucb_const=1.0):
    ucb_cr, _ = run_agent(UCB, env_kwargs, {"const": ucb_const, "init_value": 0.0, "init_count": 0},
                          horizon=horizon, n_runs=n_runs, seed=1)

    plt.figure()
    plt.plot(ucb_cr, label=f"UCB (c={ucb_const})")

    for eps in eps_list:
        gr_cr, _ = run_agent(Greedy, env_kwargs, {"epsilon": eps, "init_value": 0.0, "init_count": 0},
                             horizon=horizon, n_runs=n_runs, seed=int(1000*eps)+2)
        plt.plot(gr_cr, label=f"ε-greedy (ε={eps})")

    plt.xlabel("t")
    plt.ylabel("cumulative reward (avg over runs)")
    plt.legend()
    plt.show()

In [ ]:
env_kwargs = {"distribution": "bernoulli", "params": [0.6, 0.5]}
plot_comparison(env_kwargs, horizon=5000, n_runs=200, eps_list=(0.0, 0.01, 0.1), ucb_const=1.0)

In [ ]:
env_kwargs = {"distribution": "uniform", "params": [(0, 1), (0, 0.8)]}
plot_comparison(env_kwargs, horizon=5000, n_runs=200, eps_list=(0.01, 0.1), ucb_const=1.0)

In [ ]:
env_kwargs = {"distribution": "gaussian", "params": [(1, 1), (0, 1)]}
plot_comparison(env_kwargs, horizon=5000, n_runs=200, eps_list=(0.01, 0.1), ucb_const=1.0)

### Interpretation

- In all settings, cumulative reward grows almost linearly after a short transient phase, meaning each policy reaches an approximately constant average reward (the slope).

- **Bernoulli (0.6 vs 0.5)**: all methods are close. ε-greedy with ε=0.1 gives the best slope here, while UCB (c=1) is very competitive but slightly below. Pure greedy (ε=0) can be worse due to the risk of early commitment.

- **Uniform ((0,1) vs (0,0.8))**: curves are almost indistinguishable; the problem is easy and all policies quickly identify the best arm, so performance differences are negligible on this horizon.

- **Gaussian (N(1,1) vs N(0,1))**: UCB (c=1) performs best (or tied best) in our runs, while ε=0.1 is worse because it keeps exploring at a fixed rate. A small ε (e.g., 0.01) is close to UCB.


## Thompson sampling

Finally, we consider Thompson Sampling, where the mean rewards are considered as random and sampled according to the posterior distribution.<br>The algorithm is **Bayesian**.

## To do

* Complete the agent ``TS``, assuming Bernoulli rewards.
* Compare its performance to that of the previous algorithms. Comment the results.
* Propose a solution when the rewards are not binary.

In [ ]:
class TS(Bandit):
    """Bandit algorithm with Thompson sampling. 

    Parameters
    ----------
    environment : object of class MAB
        Environment of the agent.
    """
    
    def __init__(self, environment):
        super(TS, self).__init__(environment) 
        self.distribution = environment.distribution
            
    def get_action(self, state=None):
        """Get action with TS policy."""
        values = np.array(self.values)
        counts = np.array(self.counts)
        if self.distribution == 'bernoulli':
            successes = values * counts
            failures = counts - successes
            alpha = 1 + successes
            beta = 1 + failures
            samples = np.random.beta(alpha, beta)
        else:
            samples = np.random.normal(loc=values, scale=1.0 / np.sqrt(counts + 1.0))
        return np.argmax(samples)

In [ ]:
agent = TS(environment)

In [ ]:
def run_agent(agent_class, env_kwargs, agent_kwargs, horizon=5000, n_runs=200, seed=0):
    rng = np.random.default_rng(seed)
    cum_rewards = np.zeros(horizon)

    for r in range(n_runs):
        np.random.seed(rng.integers(0, 2**32 - 1, dtype=np.uint32).item())
        env = MAB(**env_kwargs)
        agent = agent_class(env, **agent_kwargs)
        rewards = np.array(agent.get_episode(horizon=horizon), dtype=float)
        cum_rewards += np.cumsum(rewards)

    cum_rewards /= n_runs
    return cum_rewards

env_kwargs = {"distribution": "bernoulli", "params": [0.6, 0.5]}
horizon = 5000
n_runs = 200

ts_cr  = run_agent(TS,    env_kwargs, {}, horizon=horizon, n_runs=n_runs, seed=1)
ucb_cr = run_agent(UCB,   env_kwargs, {"const": 1.0, "init_value": 0.0, "init_count": 0}, horizon=horizon, n_runs=n_runs, seed=2)
g01_cr = run_agent(Greedy,env_kwargs, {"epsilon": 0.01, "init_value": 0.0, "init_count": 0}, horizon=horizon, n_runs=n_runs, seed=3)
g1_cr  = run_agent(Greedy,env_kwargs, {"epsilon": 0.1,  "init_value": 0.0, "init_count": 0}, horizon=horizon, n_runs=n_runs, seed=4)

plt.figure()
plt.plot(ts_cr,  label="Thompson Sampling (Bernoulli)")
plt.plot(ucb_cr, label="UCB (c=1)")
plt.plot(g01_cr, label="ε-greedy (ε=0.01)")
plt.plot(g1_cr,  label="ε-greedy (ε=0.1)")
plt.xlabel("t")
plt.ylabel("cumulative reward (avg over runs)")
plt.legend()
plt.show()

### Interpretation

All curves become almost linear, meaning each algorithm reaches an approximately constant average reward (the slope).
In this experiment, Thompson Sampling achieves the highest slope (best cumulative reward), slightly ahead of ε-greedy, while UCB (c=1) is lower, suggesting it explores more than necessary on this horizon. Thompson Sampling performs well because posterior sampling naturally balances exploration and exploitation without having to tune ε.

### Answer

Use a Thompson Sampling variant with a probabilistic model adapted to the reward distribution:
- **Gaussian rewards**: assume an unknown mean with a Normal prior (and known variance), giving a Normal posterior; sample a mean from the posterior for each arm and pick the largest.
- More generally, if no conjugate model fits, use an **approximate posterior** (e.g., Normal approximation of the mean) or **bootstrap Thompson Sampling** (resample past rewards per arm and act greedily w.r.t. the resampled mean).
